# Advanced Window Functions

**Datasets:** `samples.bakehouse.sales_transactions`, `samples.bakehouse.sales_franchises`, `samples.nyctaxi.trips`

**Difficulty:** Medium

**Topics:** dense_rank, ntile, percent_rank, cume_dist, lead, rowsBetween, rangeBetween, first/last over window, multiple window specs

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

df_transactions = spark.table("samples.bakehouse.sales_transactions")
df_franchises   = spark.table("samples.bakehouse.sales_franchises")
df_trips        = spark.table("samples.nyctaxi.trips")

## Problem 1 - dense_rank vs rank

Using `df_transactions`, apply both `rank()` and `dense_rank()` over a window partitioned by `franchiseID` and ordered by `totalPrice` descending.

Show both values side by side so the difference is visible when ties occur - dense_rank never skips a number after a tie, while rank does.

**Expected output columns:**
- `franchiseID`
- `product`
- `totalPrice`
- `rnk`
- `dense_rnk`

In [ ]:
# Problem 1 - write your solution here
# Assign your result to: result_1

result_1 = None  # replace this

In [ ]:
# -- Tests for Problem 1 --
assert result_1 is not None, "result_1 is None - did you assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'rnk' in cols, "Missing column: rnk"
assert 'dense_rnk' in cols, "Missing column: dense_rnk"
assert 'franchiseid' in cols, "Missing column: franchiseID"
assert 'totalprice' in cols, "Missing column: totalPrice"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
# dense_rank never exceeds rank for the same row
violation = result_1.filter(F.col('dense_rnk') > F.col('rnk')).count()
assert violation == 0, f"Found {violation} rows where dense_rnk > rnk - dense_rank should never exceed rank"
print(f"Problem 1 passed - ({cnt} rows)")

## Problem 2 - ntile bucketing

Using `df_transactions`, apply `ntile(4)` over a window partitioned by `franchiseID` and ordered by `totalPrice` descending to assign each transaction to a quartile (1 = highest prices, 4 = lowest prices) within its franchise.

Then group by `franchiseID` and `quartile` to count how many transactions fall in each bucket.

**Expected output columns:**
- `franchiseID`
- `quartile`
- `transaction_count`

In [ ]:
# Problem 2 - write your solution here
# Assign your result to: result_2

result_2 = None  # replace this

In [ ]:
# -- Tests for Problem 2 --
assert result_2 is not None, "result_2 is None - did you assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'franchiseid' in cols, "Missing column: franchiseID"
assert 'quartile' in cols, "Missing column: quartile"
assert 'transaction_count' in cols, "Missing column: transaction_count"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
quartile_vals = [r[0] for r in result_2.select('quartile').distinct().collect()]
assert all(q in [1, 2, 3, 4] for q in quartile_vals), f"Unexpected quartile values: {quartile_vals}"
min_count = result_2.agg(F.min('transaction_count')).collect()[0][0]
assert min_count > 0, f"Expected transaction_count > 0, found min={min_count}"
print(f"Problem 2 passed - ({cnt} rows)")

## Problem 3 - percent_rank and cume_dist

Using `df_transactions`, apply both `percent_rank()` and `cume_dist()` over a window partitioned by `franchiseID` and ordered by `totalPrice` ascending.

`percent_rank` gives the relative rank as a fraction from 0 to 1, where the first row in the partition is always 0. `cume_dist` gives the fraction of rows with values less than or equal to the current row's value.

**Expected output columns:**
- `franchiseID`
- `transactionID`
- `totalPrice`
- `pct_rank`
- `cum_dist`

In [ ]:
# Problem 3 - write your solution here
# Assign your result to: result_3

result_3 = None  # replace this

In [ ]:
# -- Tests for Problem 3 --
assert result_3 is not None, "result_3 is None - did you assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'pct_rank' in cols, "Missing column: pct_rank"
assert 'cum_dist' in cols, "Missing column: cum_dist"
assert 'franchiseid' in cols, "Missing column: franchiseID"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
bounds = result_3.agg(
    F.min('pct_rank').alias('min_pr'), F.max('pct_rank').alias('max_pr'),
    F.min('cum_dist').alias('min_cd'), F.max('cum_dist').alias('max_cd')
).collect()[0]
assert bounds['min_pr'] >= 0.0, f"pct_rank below 0: {bounds['min_pr']}"
assert bounds['max_pr'] <= 1.0, f"pct_rank above 1: {bounds['max_pr']}"
assert bounds['min_cd'] >= 0.0, f"cum_dist below 0: {bounds['min_cd']}"
assert bounds['max_cd'] <= 1.0, f"cum_dist above 1: {bounds['max_cd']}"
# cume_dist >= pct_rank for every row
violation = result_3.filter(F.col('cum_dist') < F.col('pct_rank')).count()
assert violation == 0, f"Found {violation} rows where cum_dist < pct_rank"
print(f"Problem 3 passed - ({cnt} rows)")

## Problem 4 - lead function

Using `df_transactions`, apply `lead(totalPrice, 1)` over a window partitioned by `franchiseID` and ordered by `dateTime` to get each transaction's next price within the same franchise.

Also compute `price_diff` as the difference between `totalPrice` and `next_price`. The last transaction per franchise will have a null `next_price`.

**Expected output columns:**
- `franchiseID`
- `transactionID`
- `totalPrice`
- `next_price`
- `price_diff`

In [ ]:
# Problem 4 - write your solution here
# Assign your result to: result_4

result_4 = None  # replace this

In [ ]:
# -- Tests for Problem 4 --
assert result_4 is not None, "result_4 is None - did you assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'next_price' in cols, "Missing column: next_price"
assert 'price_diff' in cols, "Missing column: price_diff"
assert 'franchiseid' in cols, "Missing column: franchiseID"
assert 'totalprice' in cols, "Missing column: totalPrice"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
null_count = result_4.filter(F.col('next_price').isNull()).count()
assert null_count > 0, "Expected some null next_price values (last row per franchise has no next)"
print(f"Problem 4 passed - ({cnt} rows, {null_count} nulls in next_price as expected)")

## Problem 5 - rowsBetween (moving average)

Using `df_transactions`, calculate a 3-transaction moving average of `totalPrice` per franchise ordered by `dateTime`.

Use `rowsBetween(-2, 0)` so the window covers the current row and the 2 rows immediately preceding it (physical row-based boundary, not value-based).

**Expected output columns:**
- `franchiseID`
- `transactionID`
- `dateTime`
- `totalPrice`
- `moving_avg_3`

In [ ]:
# Problem 5 - write your solution here
# Assign your result to: result_5

result_5 = None  # replace this

In [ ]:
# -- Tests for Problem 5 --
assert result_5 is not None, "result_5 is None - did you assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'moving_avg_3' in cols, "Missing column: moving_avg_3"
assert 'franchiseid' in cols, "Missing column: franchiseID"
assert 'totalprice' in cols, "Missing column: totalPrice"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_avg = result_5.agg(F.min('moving_avg_3')).collect()[0][0]
assert min_avg > 0, f"Expected moving_avg_3 > 0 for all rows, found min={min_avg}"
null_count = result_5.filter(F.col('moving_avg_3').isNull()).count()
assert null_count == 0, f"Expected no nulls in moving_avg_3 (every row has at least itself), found {null_count}"
print(f"Problem 5 passed - ({cnt} rows)")

## Problem 6 - rangeBetween (running total with range)

Using `df_trips`, compute a cumulative sum of `fare_amount` partitioned by `pickup_zip` and ordered by `tpep_pickup_datetime`.

Use `rangeBetween(Window.unboundedPreceding, Window.currentRow)` - a value-based boundary - to build the running total.

**Expected output columns:**
- `pickup_zip`
- `tpep_pickup_datetime`
- `fare_amount`
- `running_total`

In [ ]:
# Problem 6 - write your solution here
# Assign your result to: result_6

result_6 = None  # replace this

In [ ]:
# -- Tests for Problem 6 --
assert result_6 is not None, "result_6 is None - did you assign your DataFrame?"
assert hasattr(result_6, 'columns'), "result_6 must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
assert 'running_total' in cols, "Missing column: running_total"
assert 'pickup_zip' in cols, "Missing column: pickup_zip"
assert 'fare_amount' in cols, "Missing column: fare_amount"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_6.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
# running_total must be >= fare_amount (cumulative sum never decreases)
violation = result_6.filter(F.col('running_total') < F.col('fare_amount')).count()
assert violation == 0, f"Found {violation} rows where running_total < fare_amount"
print(f"Problem 6 passed - ({cnt} rows)")

## Problem 7 - first and last over window

Using `df_transactions`, use `F.first()` and `F.last()` with `ignorenulls=True` over a window partitioned by `franchiseID`, ordered by `dateTime`, with `rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)` to see the entire partition.

Group the result down to one row per franchise showing the first and last transaction price.

**Expected output columns:**
- `franchiseID`
- `first_price`
- `last_price`

In [ ]:
# Problem 7 - write your solution here
# Assign your result to: result_7

result_7 = None  # replace this

In [ ]:
# -- Tests for Problem 7 --
assert result_7 is not None, "result_7 is None - did you assign your DataFrame?"
assert hasattr(result_7, 'columns'), "result_7 must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
assert 'franchiseid' in cols, "Missing column: franchiseID"
assert 'first_price' in cols, "Missing column: first_price"
assert 'last_price' in cols, "Missing column: last_price"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_7.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
# one row per franchise - count should equal number of distinct franchises
franchise_count = df_transactions.select('franchiseID').distinct().count()
assert cnt == franchise_count, f"Expected one row per franchise ({franchise_count}), got {cnt}"
null_first = result_7.filter(F.col('first_price').isNull()).count()
null_last = result_7.filter(F.col('last_price').isNull()).count()
assert null_first == 0, f"Found {null_first} nulls in first_price"
assert null_last == 0, f"Found {null_last} nulls in last_price"
print(f"Problem 7 passed - ({cnt} rows, one per franchise)")

## Problem 8 - multiple window specs in one query

Using `df_transactions`, apply three different window specifications in a single `.withColumn` chain:

1. `row_number()` over `(PARTITION BY franchiseID ORDER BY totalPrice DESC)` - aliased as `rn`
2. `sum(totalPrice)` over `(PARTITION BY franchiseID)` (no ordering) - aliased as `franchise_total`
3. `avg(totalPrice)` over `(PARTITION BY franchiseID ORDER BY dateTime ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)` - aliased as `moving_avg`

After computing all three, filter to keep only rows where `rn <= 5` (top 5 highest-priced transactions per franchise).

**Expected output columns:**
- `franchiseID`
- `transactionID`
- `totalPrice`
- `franchise_total`
- `moving_avg`
- `rn`

In [ ]:
# Problem 8 - write your solution here
# Assign your result to: result_8

result_8 = None  # replace this

In [ ]:
# -- Tests for Problem 8 --
assert result_8 is not None, "result_8 is None - did you assign your DataFrame?"
assert hasattr(result_8, 'columns'), "result_8 must be a Spark DataFrame"
cols = [c.lower() for c in result_8.columns]
assert 'franchiseid' in cols, "Missing column: franchiseID"
assert 'transactionid' in cols, "Missing column: transactionID"
assert 'totalprice' in cols, "Missing column: totalPrice"
assert 'franchise_total' in cols, "Missing column: franchise_total"
assert 'moving_avg' in cols, "Missing column: moving_avg"
assert 'rn' in cols, "Missing column: rn"
assert len(cols) == 6, f"Expected exactly 6 columns, got {len(cols)}: {cols}"
cnt = result_8.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
max_rn = result_8.agg(F.max('rn')).collect()[0][0]
assert max_rn <= 5, f"Expected rn <= 5 after filter, found max rn={max_rn}"
min_total = result_8.agg(F.min('franchise_total')).collect()[0][0]
assert min_total > 0, f"Expected franchise_total > 0, found min={min_total}"
print(f"Problem 8 passed - ({cnt} rows)")